In [1]:
from pathlib import Path

import util
from util import workflow

browser = False
file = util.notebook_file() if util.is_notebook() else __file__
tag = util.file_tag(file)
root_path = Path("..")
data_path = util.data_path(root_path)

In [2]:
# Build
from automol.graph import enum

import automech
from automech.schema import Species

mech0 = automech.io.read(data_path / "full_raw.json")
mech = automech.from_smiles(spc_smis=["C1=CCCC1"], src_mech=mech0)
#  - add OH addition to *ene*
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.pi2_addition,
    rcts_=[None, "[OH]"],
    spc_col_=Species.smiles,
    src_mech=mech0,
)
#  - add 1,2 H-migrations (1,3 should be negligible)
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.h_migration,
    spc_col_=Species.smiles,
    src_mech=mech0,
    repeat=2
)
#  - add ring beta scissions
mech = automech.enumerate_reactions(
    mech,
    enum.ReactionSmarts.ring_beta_scission,
    src_mech=mech0,
)

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

In [3]:
# Write
workflow.write(mech=mech, tag=tag, root_path=root_path, browser=browser)


Finalizing build for...
Mechanism(
  reactions=DataFrame(
    shape: (13, 5)
    ┌────────────────┬────────────────┬───────────┬──────────────────────────┬─────────────────────────┐
    │ reactants      ┆ products       ┆ formula   ┆ rate                     ┆ colliders               │
    │ ---            ┆ ---            ┆ ---       ┆ ---                      ┆ ---                     │
    │ list[str]      ┆ list[str]      ┆ struct[3] ┆ struct[7]                ┆ struct[11]              │
    ╞════════════════╪════════════════╪═══════════╪══════════════════════════╪═════════════════════════╡
    │ ["C5H8(522)",  ┆ ["C5H9O(852)"] ┆ {5,9,1}   ┆ {{1.0,0.0,0.0},true,"Plo ┆ {null,null,null,null,nu │
    │ "OH(4)"]       ┆                ┆           ┆ g",nul…                  ┆ ll,null…                │
    │ ["C5H9O(852)"] ┆ ["S(1248)"]    ┆ {5,9,1}   ┆ {{6.1637e8,1.3,41.15},tr ┆ {null,null,null,null,nu │
    │                ┆                ┆           ┆ ue,"Co…                  ┆ ll,

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]


Writing mechanism...
../data/A_rh-oh_p2v2_raw.json
../data/A_rh-oh_p2v2.json
../data/mechanalyzer/A_rh-oh_p2v2.dat
../data/mechanalyzer/A_rh-oh_p2v2.csv


  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]


Stereoexpansion errors:


In [4]:
# Read
workflow.read(tag=tag, root_path=root_path)


Reading mechanisms...

Adding calculated thermo...

Adding calculated rates...

Expanding and updating parent...


  0%|          | 0/2052 [00:00<?, ?it/s]


Writing mechanism...
../data/A_rh-oh_p2v2_calc.json
../data/full_A_rh-oh_p2v2_control.json
../data/full_A_rh-oh_p2v2_calc.json
../data/chemkin/full_A_rh-oh_p2v2_control.dat
../data/chemkin/full_A_rh-oh_p2v2_calc.dat

Compare calculated mechanism to parent mechanism...

1. Original (compare):
C5H9O(859) = CPTOJ(880)                   2.740E+09      0.000      6.900
C5H8(522) + OH(4) = C5H9O(852)            1.000E+00      0.000      0.000
    PLOG  /                      0.01000  3.650E+77     -20.00      33.87/
    PLOG  /                      0.01000  9.700E+59     -15.51      12.90/
    PLOG  /                       0.1000  1.380E+68     -17.01      27.91/
    PLOG  /                       0.1000  1.800E+56     -14.04      12.95/
    PLOG  /                        1.000  2.600E+59     -14.17      23.08/
    PLOG  /                        1.000  2.070E+50     -12.04      11.49/
    PLOG  /                        10.00  1.010E+54     -12.23      22.98/
    PLOG  /                      

In [5]:
# # Simulate
# workflow.simulate(full_tag=f"full_{tag}_calc", root_path=root_path)
# workflow.simulate(full_tag=f"full_{tag}_control", root_path=root_path)